# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Growing content is longer and younger

On page 6, the paper reports that growing pages average 3,180 words and 184 days old, while declining pages average 2,311 words and 230 days old. The trend label is based on the change in impressions between the most recent 30 days and the previous 30 days.

**Methodology question:** Do these differences remain when pages are compared within the same client and similar search-demand groups? A pooled comparison may partly reflect differences between clients, topics, seasonality, or baseline demand. I would also ask whether the pages were evaluated on a later time period or whether the same observation window was used to define both the trend label and the page characteristics.

**Interpretation:** This finding provides observed evidence of an association between content length, age, and trend direction. It does not establish that increasing word count or refreshing a page causes growth.

### Finding 2 — The freshness multiplier

On page 9, the paper reports that pages older than 365 days and refreshed within 30 days had higher average health scores and impressions than comparable stale pages. It also notes that the 361+ day growth-to-decline ratio is unstable because the declining group contains only one page in that subset.

**Methodology question:** How was a refresh identified, and were refreshed pages compared with matched non-refreshed pages from the same clients, age bands, topics, and baseline traffic levels? Pages selected for refresh may already differ from pages left untouched. A time-aware pre-refresh and post-refresh comparison with a suitable control group would be needed to support a causal claim.

**Interpretation:** The result is a useful directional observation for deciding which pages to investigate. The disclosed validation does not by itself prove that refreshing a page produced the measured improvement.

In [1]:
import pandas as pd
from IPython.display import display

paper_audit = pd.DataFrame([
    {
        "paper_finding": "Growing pages are longer and younger",
        "label_source": "Latest 30-day impressions compared with the previous 30 days",
        "methodology_question": (
            "Does the relationship hold within clients and similar demand groups, "
            "and was it checked on a later period?"
        ),
        "safe_interpretation": "Observed association, not causal evidence"
    },
    {
        "paper_finding": "Recently refreshed old pages show stronger metrics",
        "label_source": "Days since update plus observed health and impression measures",
        "methodology_question": (
            "Were refreshed pages compared with matched controls using a "
            "time-aware pre/post design?"
        ),
        "safe_interpretation": "Directional decision-support evidence"
    }
])

display(paper_audit)

,paper_finding,label_source,methodology_question,safe_interpretation
0,Growing pages are longer and younger,Latest 30-day impressions compared with the pr...,Does the relationship hold within clients and ...,"Observed association, not causal evidence"
1,Recently refreshed old pages show stronger met...,Days since update plus observed health and imp...,Were refreshed pages compared with matched con...,Directional decision-support evidence


## 2. My model under an honest split (before/after)

### Before-and-after validation design

The **before** diagnostic uses a row-random 75/25 split. This split can place pages belonging to the same client in both training and test data, so client-specific patterns may make the result optimistic.

The **after** diagnostic uses `GroupShuffleSplit` with `client_id` as the group. Entire clients are held out, and no client appears in both training and test data. This is more honest for the question: “Can the scoring method provide useful decision support for clients not seen during training?”

Both versions use the same Logistic Regression pipeline, feature set, test fraction, and metrics. Precision@50 is the primary queue metric because the practical output is a short ranked review list. Average Precision and ROC AUC are included as supporting measures.

The two test sets have different compositions and base rates, so their metric difference is directional rather than a perfectly controlled estimate.

In [2]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    roc_auc_score
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from IPython.display import display


# ------------------------------------------------------------
# Load the anonymized starter dataset.
# ------------------------------------------------------------
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    REPO_PATH = Path("/content/flyrank-ml-internship-starter")

    if not REPO_PATH.exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git",
                str(REPO_PATH)
            ],
            check=True
        )

    DATA_PATH = REPO_PATH / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

# Proxy target used in Week 5.
df["is_declining_proxy"] = (
    df["trend_direction"].astype(str).str.lower().eq("down")
).astype(int)

# Position zero means unavailable in this dataset.
df["avg_position_clean"] = df["avg_position"].replace(0, np.nan)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])

FEATURE_COLS = [
    "log_impressions_90d",
    "log_sessions_90d",
    "ctr",
    "avg_position_clean",
    "content_age_days",
    "days_since_last_update",
    "days_with_impressions"
]

TARGET = "is_declining_proxy"
GROUP = "client_id"

X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()
groups = df[GROUP].copy()


def make_model():
    return Pipeline([
        (
            "prepare",
            ColumnTransformer([
                (
                    "numeric",
                    Pipeline([
                        (
                            "impute",
                            SimpleImputer(
                                strategy="median",
                                add_indicator=True
                            )
                        ),
                        ("scale", StandardScaler())
                    ]),
                    FEATURE_COLS
                )
            ], remainder="drop")
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])


def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())


def fit_and_measure(train_idx, test_idx):
    model = make_model()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])

    scores = model.predict_proba(X.iloc[test_idx])[:, 1]
    y_test = y.iloc[test_idx].to_numpy()

    metrics = {
        "test_rows": len(test_idx),
        "test_base_rate": y_test.mean(),
        "precision_at_10": precision_at_k(y_test, scores, 10),
        "precision_at_20": precision_at_k(y_test, scores, 20),
        "precision_at_50": precision_at_k(y_test, scores, 50),
        "average_precision": average_precision_score(y_test, scores),
        "roc_auc": roc_auc_score(y_test, scores)
    }

    return model, scores, metrics


all_indices = np.arange(len(df))

# Before: row-random split.
random_train, random_test = train_test_split(
    all_indices,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# After: entire clients held out.
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

group_train, group_test = next(
    group_splitter.split(X, y, groups=groups)
)

random_model, random_scores, random_metrics = fit_and_measure(
    random_train,
    random_test
)

grouped_model, grouped_scores, grouped_metrics = fit_and_measure(
    group_train,
    group_test
)

comparison = pd.DataFrame([
    {"validation": "Before: row-random", **random_metrics},
    {"validation": "After: held-out clients", **grouped_metrics}
])

metric_columns = [
    "test_base_rate",
    "precision_at_10",
    "precision_at_20",
    "precision_at_50",
    "average_precision",
    "roc_auc"
]

comparison[metric_columns] = comparison[metric_columns].round(3)

train_clients = set(groups.iloc[group_train])
test_clients = set(groups.iloc[group_test])

print("Grouped-split client overlap:", len(train_clients & test_clients))
display(comparison)

Grouped-split client overlap: 0


,validation,test_rows,test_base_rate,precision_at_10,precision_at_20,precision_at_50,average_precision,roc_auc
0,Before: row-random,7500,0.542,0.8,0.90,0.88,0.687,0.678
1,After: held-out clients,7115,0.517,0.9,0.85,0.84,0.618,0.608


## 3. Leakage audit

I checked the final feature list for four risks:

1. **Direct label leakage:** `trend_direction`, `trend_pct`, and `is_declining_proxy` are not model inputs. `trend_pct` directly determines the proxy label, so including it would make the score invalid.
2. **Identifiers:** `content_id` and `client_id` are used only for identification, grouping, and validation. They are not predictive features.
3. **Product flags:** No existing FlyRank action or product flag is used as a model feature.
4. **Window overlap:** The starter dataset contains trailing-90-day summaries, while the proxy label compares recent 30-day periods. These windows can overlap. This is not direct label-column leakage, but it limits the claim: the model measures association with the current trend proxy and should not be described as a validated future forecast.

The deliberate experiment below adds `trend_pct`. Its near-perfect result demonstrates why label-derived fields must be deleted. The final retained model is the grouped model using only `FEATURE_COLS`.

For a later capstone version, I would construct features from a period ending before the decision date and construct the label from a strictly later outcome period.

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


FORBIDDEN_FIELDS = {
    "trend_direction",
    "trend_pct",
    "is_declining_proxy",
    "content_id",
    "client_id"
}

print(
    "Forbidden fields found in honest feature set:",
    sorted(FORBIDDEN_FIELDS.intersection(FEATURE_COLS))
)

assert FORBIDDEN_FIELDS.isdisjoint(FEATURE_COLS)


# ------------------------------------------------------------
# Deliberately add one label-derived field.
# ------------------------------------------------------------
LEAKY_COLS = FEATURE_COLS + ["trend_pct"]

leaky_model = Pipeline([
    (
        "impute",
        SimpleImputer(strategy="median", add_indicator=True)
    ),
    ("scale", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

leaky_model.fit(
    df.iloc[group_train][LEAKY_COLS],
    y.iloc[group_train]
)

leaky_scores = leaky_model.predict_proba(
    df.iloc[group_test][LEAKY_COLS]
)[:, 1]

grouped_y_test = y.iloc[group_test].to_numpy()

leakage_comparison = pd.DataFrame([
    {
        "model": "Honest grouped model",
        "precision_at_50": precision_at_k(
            grouped_y_test, grouped_scores, 50
        ),
        "average_precision": average_precision_score(
            grouped_y_test, grouped_scores
        ),
        "roc_auc": roc_auc_score(
            grouped_y_test, grouped_scores
        )
    },
    {
        "model": "Invalid model with trend_pct",
        "precision_at_50": precision_at_k(
            grouped_y_test, leaky_scores, 50
        ),
        "average_precision": average_precision_score(
            grouped_y_test, leaky_scores
        ),
        "roc_auc": roc_auc_score(
            grouped_y_test, leaky_scores
        )
    }
]).round(3)

display(leakage_comparison)

print(
    "trend_pct is label-derived and has been removed. "
    "Final retained features:"
)
print(FEATURE_COLS)


# ------------------------------------------------------------
# Inspect real errors from the honest grouped model.
# ------------------------------------------------------------
error_review = df.iloc[group_test][[
    "content_id",
    "trend_direction",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]].copy()

error_review["actual_decline"] = grouped_y_test
error_review["predicted_probability"] = grouped_scores
error_review["predicted_decline"] = (
    error_review["predicted_probability"] >= 0.50
).astype(int)

false_positives = error_review[
    (error_review["predicted_decline"] == 1) &
    (error_review["actual_decline"] == 0)
].copy()

false_negatives = error_review[
    (error_review["predicted_decline"] == 0) &
    (error_review["actual_decline"] == 1)
].copy()

false_positives["error_type"] = "False positive"
false_positives["why_it_may_be_wrong"] = (
    "Signals resemble decline, but the observed proxy label is not down."
)

false_negatives["error_type"] = "False negative"
false_negatives["why_it_may_be_wrong"] = (
    "The observed decline was not captured strongly by the selected signals."
)

error_examples = pd.concat([
    false_positives.nlargest(3, "predicted_probability"),
    false_negatives.nsmallest(3, "predicted_probability")
])

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

display(error_examples[[
    "content_id",
    "error_type",
    "actual_decline",
    "predicted_probability",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "why_it_may_be_wrong"
]].round({
    "predicted_probability": 3,
    "ctr": 3,
    "avg_position": 2
}))

Forbidden fields found in honest feature set: []


,model,precision_at_50,average_precision,roc_auc
0,Honest grouped model,0.84,0.618,0.608
1,Invalid model with trend_pct,1.00,1.000,1.000


trend_pct is label-derived and has been removed. Final retained features:
['log_impressions_90d', 'log_sessions_90d', 'ctr', 'avg_position_clean', 'content_age_days', 'days_since_last_update', 'days_with_impressions']
False positives: 1604
False negatives: 1376


,content_id,error_type,actual_decline,predicted_probability,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,why_it_may_be_wrong
27993,content_26d48a980581,False positive,0,0.802,1266,0.00,4.6,106,106,"Signals resemble decline, but the observed pro..."
25298,content_10c16c9b435a,False positive,0,0.800,2558,0.12,7.3,106,8,"Signals resemble decline, but the observed pro..."
15138,content_f55c4e2464ad,False positive,0,0.795,869,0.35,13.5,96,20,"Signals resemble decline, but the observed pro..."
27271,content_7bc32bc1df59,False negative,1,0.009,1,0.00,0.0,238,92,The observed decline was not captured strongly...
24849,content_2f002563e9cd,False negative,1,0.145,17,0.00,5.2,502,20,The observed decline was not captured strongly...
17690,content_c268b1716236,False negative,1,0.162,3,0.00,41.7,502,20,The observed decline was not captured strongly...


## 4. Claim rewrite
### Original claim — too strong

“The Logistic Regression model predicts declining pages accurately and proves that these pages should be refreshed.”

### Rewritten safe claim

On one fixed held-out-client split of the anonymized starter dataset, Logistic Regression achieved a measured Precision@50 of approximately 0.84. This was higher than the Week-4 stale-and-visible baseline on its corresponding evaluation.

The result is directional evidence that the selected signals can support a ranked page-review queue for held-out clients in this dataset. It does not prove that a refresh will improve performance, establish a causal relationship, or guarantee performance in a future time period.

The proxy label and some trailing-window features also have temporal overlap. Therefore, I describe this version as decision-support analysis of an observed trend proxy rather than validated future-decline prediction.

In [4]:
random_p50 = float(
    comparison.loc[
        comparison["validation"] == "Before: row-random",
        "precision_at_50"
    ].iloc[0]
)

grouped_p50 = float(
    comparison.loc[
        comparison["validation"] == "After: held-out clients",
        "precision_at_50"
    ].iloc[0]
)

print(f"Observed row-random Precision@50: {random_p50:.3f}")
print(f"Observed held-out-client Precision@50: {grouped_p50:.3f}")

print(
    "\nFinal interpretation: the grouped result provides directional "
    "decision-support evidence on this dataset. It is not proof of "
    "future performance or causal refresh impact."
)

Observed row-random Precision@50: 0.880
Observed held-out-client Precision@50: 0.840

Final interpretation: the grouped result provides directional decision-support evidence on this dataset. It is not proof of future performance or causal refresh impact.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.